In [ ]:
# 📦 Kaggle environment boilerplate — lists available input files and sets up kagglehub access.
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

## 📥 Importing Necessary Libraries
Loading pandas, numpy, matplotlib, and seaborn — the core stack used throughout this notebook for data handling, numerical operations, and visualization.

In [ ]:
# Core data manipulation and visualization libraries used throughout the notebook.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 📂 Loading the Data
Reading the `train`, `test`, and `sample_submission` CSV files for the bank customer churn prediction competition (`mlp-kaggle-assignment-2-may-2026-term`).

In [ ]:
# Loading the competition's train/test/sample_submission CSVs.
train_data=pd.read_csv('/kaggle/input/competitions/mlp-kaggle-assignment-2-may-2026-term/train.csv')
test_data=pd.read_csv('/kaggle/input/competitions/mlp-kaggle-assignment-2-may-2026-term/test.csv')
sample_submission=pd.read_csv('/kaggle/input/competitions/mlp-kaggle-assignment-2-may-2026-term/sample_submission.csv')

In [ ]:
# Working on a deep copy so the original train_data stays untouched for reference.
df=train_data.copy(deep=True)

In [ ]:
# Random 7-row sample to eyeball the raw data.
df.sample(7)

## 📏 Shape of Our Data
Checking the number of rows and columns to get a sense of dataset size before diving into exploration.

In [ ]:
# Rows x columns.
df.shape

📌 **Observation:** We have 90,000 rows and 14 columns.

## 🔎 Checking the Datatypes
Inspecting column dtypes and non-null counts to spot numeric vs. categorical features and flag any columns that may need type conversion.

In [ ]:
# dtypes + non-null counts for every column.
train_data.info()

### 🧹 Dropping the `id` Column
The `id` column is a row identifier with no predictive value — dropping it early keeps the feature set clean and avoids introducing noise into the model.

In [ ]:
# Dropping the id column — guarded so re-running the cell doesn't error if it's already gone.
# dropping the id column
if 'id' in df.columns:
     
    df=df.drop(columns=['id'])
    print('id column removed')

else:
    print('id is already removed')

## 📊 Descriptive Statistics — Numerical Columns
Summary stats (mean, std, min/max, quartiles) for numeric features to understand scale, spread, and potential outliers.

In [ ]:
# Summary statistics for numeric columns (count, mean, std, min, quartiles, max).
df.describe()

## 📊 Descriptive Statistics — Categorical Columns
Summary stats (count, unique, top, freq) for categorical/object-type features.

In [ ]:
# Summary statistics for categorical/object columns (count, unique, top, freq).
df.describe(include=['object'])

## 🕳️ Identifying Missing Values
Counting nulls per column, sorted descending, to prioritize which columns need an imputation strategy.

In [ ]:
# Missing value counts per column, highest first.
df.isna().sum().sort_values(ascending=False)

📌 **Observation:** `credit_score`, `country`, `acc_balance`, and `prod_count` have missing values — these will be handled later with tailored imputation strategies for each column.

## 🧾 Auditing for Duplicates
Checking whether any rows are fully duplicated, which would need to be removed to avoid biasing the model.

In [ ]:
# Checking whether any rows are exact duplicates across all columns.
print('Duplicates in training data',df.duplicated().sum())

📌 **Observation:** There are no duplicate rows — no need to remove anything here.

In [ ]:
# Number of unique values per column, ascending — a quick cardinality check.
# checking for no of uniques values in every column
df.nunique().sort_values()

## ⚠️ Checking for Unusual Values (Other Than NaN)
Beyond missing values, we also want to catch invalid entries — e.g. typos, impossible values, or inconsistent category labels — that wouldn't show up as `NaN` but would still corrupt the model.

In [ ]:
# Value counts (including NaN) for the low-cardinality columns to spot anything unexpected.
# first for columns with low cardinality
columns_to_check = ['gender', 'country', 'has_card', 'is_active', 'prod_count', 'exit_status','tenure']
for column in columns_to_check:
    print(df[column].value_counts(dropna=False))
    print('-'*30)

📌 **Observation:** All low-cardinality columns look clean — no unusual values found.

In [ ]:
# Re-checking describe().T for high-cardinality/numeric columns to catch out-of-range values.
# Now for high cardinality columns we will check numerical statistics
df.describe().T


📌 **Observation:** No unusual statistics found; min/max values for all numeric columns fall within a reasonable range. The `last_name` column looks like a strong candidate to drop later.

## 🕵️ Analyzing the `customer_id` Column — Something's Off
Digging deeper into `customer_id` since a customer identifier repeating across rows with *different* attributes would suggest it isn't a reliable unique key.

In [ ]:
# Grouping by customer_id to inspect whether it behaves like a unique identifier.
grouped = df.groupby('customer_id')

In [ ]:
# Filtering to customer_ids that appear more than once.
multi_entries = grouped.filter(lambda x: len(x) > 1)
print('Customer id with multiple entries',multi_entries.count()['customer_id'])

📌 **Observation:** There are 82,608 `customer_id` values with multiple entries — a strong sign this column doesn't uniquely identify customers as expected.

In [ ]:
# Inspecting all rows for one specific customer_id to see what differs between duplicate entries.
# getting information from one particular customer id
grouped.get_group(15642816)

### 🚨 Conclusion: `customer_id` Is Noise
Multiple rows share the same `customer_id` but have different `last_name`, `acc_balance`, `country`, etc. This confirms `customer_id` is synthetically generated and carries no real identifying signal — so it's dropped along with `last_name`.

In [ ]:
# Dropping customer_id (proven to be non-unique/noisy) and last_name (no predictive value) — guarded against re-running.
# dropping the customer_id and last name  column
discarded_columns=['customer_id','last_name']
for col in discarded_columns:
    if col in df.columns:
         
        df=df.drop(columns=[col])
        print(f'{col} column removed')
    
    else:
        print(f'{col} is already removed')

In [ ]:
# Sanity check on the dataframe after dropping columns.
df.head()

## 📦 Checking and Handling Outliers
Using the IQR method to flag outliers in numeric columns before deciding whether they need to be treated or can be safely left alone.

In [ ]:
# Splitting columns into numeric and categorical lists for targeted preprocessing.
# Seperating numerical and categorical columns
num_cols=df.select_dtypes(include=['int64','float64']).columns.to_list()
cat_cols=df.select_dtypes(include=['object']).columns.to_list()

In [ ]:
# Another quick look at the cleaned dataframe.
df.head()

In [ ]:
# IQR-based outlier detection: for each numeric column, compute Q1/Q3/IQR, define bounds, and count/percentage of values outside them.
outlier_summary={}

for cols in num_cols:
    # Calculating Q1, Q3, and IQR
    Q1=df[cols].quantile(.25)
    Q3=df[cols].quantile(.75)

    #Calculating IQR
    IQR=Q3-Q1

    # Defining Upper and Lower bounds
    lower_bound = Q1 - 1.5*IQR
    upper_bound = Q3 + 1.5*IQR 

    valid_data= df[cols].dropna()
    total_count = len(valid_data)

    # Identifying Outliers
    outliers = valid_data[(valid_data<lower_bound) | (valid_data>upper_bound)]
    outlier_count=len(outliers)

    # Calculating Outlier percentage
    outlier_percentage = (outlier_count/total_count)*100 if total_count>0 else 0

    # storing results
    outlier_summary[cols] = {
        'Total Valid Rows': total_count,
        'Outlier Count': outlier_count,
        'Outlier Percentage (%)': round(outlier_percentage, 3)
    }

outlier_df = pd.DataFrame(outlier_summary).T
display(outlier_df)

📌 **Observation:** Only `age` and `credit_score` contain a small percentage of valid outliers. Since tree-based models are fairly robust to outliers and the affected percentage is small, we leave them as-is rather than removing or capping them.

## 📈 Data Visualization
Visual exploration of the target variable and key features to understand distributions, class balance, and relationships with churn.

### 🎯 Countplot for the Target Column (`exit_status`)
Visualizing how many customers churned vs. stayed.

In [ ]:
# Countplot of the target variable with count labels on each bar.


ax = sns.countplot(data=df, x='exit_status')

# Adding count labels
for container in ax.containers:
    ax.bar_label(container)

plt.show()

### ⚖️ Observation: Clear Class Imbalance
The target classes are not evenly distributed — this will need to be accounted for during modeling (e.g. `class_weight='balanced'`, `scale_pos_weight`).

In [ ]:
# 2x2 grid: credit_score distribution, account balance distribution, product count counts, and country counts.
fig,axes=plt.subplots(2,2,figsize=(25,15))
sns.histplot(data=df,x='credit_score',kde=True,ax=axes[0,0])
axes[0,0].set_title('Distribution of credit_score')

sns.histplot(data=df,x='acc_balance',kde=True,ax=axes[0,1])
axes[0,1].set_title('Distribution of account balance')

ax=sns.countplot(data=df,x='prod_count',ax=axes[1,0])
axes[1,0].set_title('Count plot of Product count')

for container in ax.containers:
    ax.bar_label(container)

ax=sns.countplot(data=df,x='country',ax=axes[1,1])
axes[1,1].set_title('Countplot of country')
for container in ax.containers:
    ax.bar_label(container)

### 📌 Distribution Insights & Imputation Strategy
- `credit_score` is roughly normally distributed with a few odd spikes → **mean/median imputation** is a good fit.
- `acc_balance` is zero-inflated → imputing with a flat mean/median would be misleading, so imputing with **zero** (or a distribution-aware approach) makes more sense.
- `prod_count` is a discrete count where most customers have 2 products → **mode imputation** fits best.
- `country` benefits from **proportional random imputation** to preserve the original category distribution/variance rather than collapsing everyone into the most frequent country.

In [ ]:
# Side-by-side churn rate comparison by country and by gender, with data labels on the bars.

plt.figure(figsize=(12, 5))

# Subplot 1: Churn by Country
ax1 = plt.subplot(1, 2, 1)

sns.barplot(x='country', y='exit_status', data=df, ax=ax1)
plt.title('Churn Proportion by Country')
plt.ylabel('Churn Rate')

# Adding data labels on top of the bars for Country
for container in ax1.containers:
    ax1.bar_label(container, fmt='%.3f', padding=3)

# Subplot 2: Churn by Gender
ax2 = plt.subplot(1, 2, 2)
sns.barplot(x='gender', y='exit_status', data=df, ax=ax2)
plt.title('Churn Proportion by Gender')
plt.ylabel('Churn Rate')

# Adding data labels on top of the bars for Gender
for container in ax2.containers:
    ax2.bar_label(container, fmt='%.3f', padding=3)

plt.tight_layout()
plt.show()

📌 **Observation:** Germany shows a much higher churn rate (0.381) compared to France (0.166) and Spain (0.171). Females (0.280) churn significantly more than males (0.159) — both `country` and `gender` look like useful churn signals.

In [ ]:
# 2x2 grid comparing age and account balance distributions (boxplot + KDE) split by exit_status.


fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- Age Visualizations ---
# Boxplot
sns.boxplot(x='exit_status', y='age', data=df, ax=axes[0, 0])
axes[0, 0].set_title('Boxplot: Age by Exit Status')

# KDE Plot
sns.kdeplot(data=df, x='age', hue='exit_status', fill=True, common_norm=False, ax=axes[0, 1])
axes[0, 1].set_title('KDE: Age by Exit Status')

# --- Account Balance Visualizations ---
# Boxplot
sns.boxplot(x='exit_status', y='acc_balance', data=df, ax=axes[1, 0])
axes[1, 0].set_title('Boxplot: Account Balance by Exit Status')

# KDE Plot
sns.kdeplot(data=df, x='acc_balance', hue='exit_status', fill=True, common_norm=False, ax=axes[1, 1])
axes[1, 1].set_title('KDE: Account Balance by Exit Status')

plt.tight_layout()
plt.show()

📌 **Observation:** Older customers (45–50) show a sharp rise in churn compared to the 30–40 age group, making age a strong predictor. Zero-balance accounts tend to stay, while high balances correlate with higher churn risk. Together, age and account balance emerge as highly discriminating features the model should emphasize.

In [ ]:
# Pearson correlation matrix among numeric columns.
correlation=df[num_cols].corr()
correlation

In [ ]:
# Correlation heatmap for a quick visual read of the matrix above.
# Coorelation heatmap
sns.heatmap(correlation,cmap='coolwarm')

## 🌍 Imputing the `country` Column with a Custom Proportional Imputer
Rather than filling missing `country` values with the single most frequent category (which would distort the class distribution), a custom function randomly assigns missing values using the *existing* category probabilities — preserving the original proportions.

In [ ]:
# Custom imputer: fills missing values in a categorical column by randomly sampling from the column's OWN existing category distribution,
# preserving the original class proportions instead of collapsing everything into the mode.
import numpy as np

def proportional_imputation(dataframe, column_name):
    # Calculating the probabilities of existing categories
    # normalize=True returns the relative frequencies (percentages)
    value_counts = dataframe[column_name].value_counts(normalize=True)
    categories = value_counts.index
    probabilities = value_counts.values
    
    
    # Identifying the number of missing values to fill
    missing_count = dataframe[column_name].isnull().sum()
    
    # Generating random replacements  weighted by the probabilities
    
    replacements = np.random.choice(categories, size=missing_count, p=probabilities)
    
    
    # Locating the exact indices of missing values and assigning the replacements
    
    dataframe.loc[dataframe[column_name].isnull(), column_name] = replacements
    
    
    return dataframe

# Applying the function to the 'country' column
df = proportional_imputation(df, 'country')

# Verifying the new distribution matches the original
print("New Country Distribution:\n", df['country'].value_counts(normalize=True))


In [ ]:
# Custom imputer for the zero-inflated acc_balance column: preserves the observed zero-rate and samples remaining missing values
# from the actual distribution of non-zero balances (reproducible via a fixed seed).
def impute_acc_balance(data, zero_rate, nonzero_values, seed=42):
    data = data.copy()
    rng = np.random.RandomState(seed)
    mask = data['acc_balance'].isnull()
    n_missing = mask.sum()
    is_zero = rng.random(n_missing) < zero_rate
    imputed = np.where(is_zero, 0, rng.choice(nonzero_values, size=n_missing))
    data.loc[mask, 'acc_balance'] = imputed
    return data

zero_rate = (df['acc_balance'].dropna() == 0).mean()
nonzero_values = df.loc[df['acc_balance'] > 0, 'acc_balance'].dropna().values
df = impute_acc_balance(df, zero_rate, nonzero_values, seed=42)

## 🛠️ Feature Engineering
Before splitting the data, a few derived features are engineered from the cleaned columns (zero-balance flag, senior-risk flag, engagement score, etc.) to give the models more predictive signal.

In [ ]:
# Feature engineering: derives zero-balance flag, senior-age-risk flag, engagement score (has_card + is_active),
# inactive-with-products flag, Germany+Female interaction flag, and two ratio features (tenure/age, products/tenure).
# prod_count is filled with the training-set mode before being used in these calculations.
def engineer_features(data, prod_count_mode):
    data = data.copy()
    prod_count_filled = data['prod_count'].fillna(prod_count_mode)
    data['zero_balance_flag'] = (data['acc_balance'] == 0).astype(int)
    data['is_senior_risk'] = ((data['age'] >= 45) & (data['age'] <= 50)).astype(int)
    data['engagement_score'] = data['has_card'] + data['is_active']
    data['inactive_with_products'] = ((data['is_active'] == 0) & (prod_count_filled > 1)).astype(int)
    data['germany_female'] = ((data['country'] == 'Germany') & (data['gender'] == 'Female')).astype(int)
    data['tenure_per_age'] = data['tenure'] / (data['age'] + 1)
    data['products_per_tenure'] = prod_count_filled / (data['tenure'] + 1)
    return data

prod_count_mode = df['prod_count'].mode()[0]
df = engineer_features(df, prod_count_mode)

In [ ]:
# Preview the dataframe after feature engineering.
df.head()

In [ ]:
# Stratified 80/20 train-validation split to preserve the target class ratio in both sets.
from sklearn.model_selection import train_test_split
X=df.drop('exit_status',axis=1)
y=df['exit_status']

X_train,X_val,y_train,y_val = train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)

## ⚙️ Building the Preprocessing Pipeline
Assembling column-specific imputation and encoding steps into a single `ColumnTransformer`, wrapped in a `Pipeline` with a final scaler — ensuring the exact same preprocessing is applied consistently to train, validation, and test data.

In [ ]:
# Preprocessing building blocks: imputers, encoders/scalers, feature selection, and ColumnTransformer.
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MaxAbsScaler,OneHotEncoder,MinMaxScaler,StandardScaler,RobustScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.compose import ColumnTransformer

In [ ]:
# Quick check of the dataframe just before building the pipeline.
df.head()

In [ ]:
# Per-column preprocessing pipelines:
# - gender: mode-impute + one-hot encode (drop first to avoid the dummy trap)
# - country: one-hot encode only (already imputed earlier via the custom proportional imputer)
# - credit_score: mean-impute
# - prod_count: mode-impute
# (account_balance already imputed earlier via the custom zero-inflated imputer, so no transformer needed here)
gender_transformer=Pipeline([
    ('impute',SimpleImputer(strategy='most_frequent')),
    ('encode',OneHotEncoder(sparse_output=False,drop='first'))
])

# Not imputing because already impuded using custom imputer
country_transformer=Pipeline([
    # ('impute',SimpleImputer(strategy='most_frequent')),
    ('encode',OneHotEncoder(sparse_output=False))
])

credit_score_transformer=Pipeline([
    ('impute',SimpleImputer(strategy='mean')),
])

# account_balance_transformer=Pipeline([
#     ('impute',SimpleImputer(strategy='constant',fill_value=0))
# ])

prod_count_transformer=Pipeline([
    ('impute',SimpleImputer(strategy='most_frequent'))
])


In [ ]:
# Combining all column transformers into one ColumnTransformer (untouched columns pass through unchanged),
# then wrapping it with a StandardScaler in a single end-to-end pipeline. Output is kept as a pandas DataFrame for readability.
# Final Transformation

from sklearn import set_config

# Set the global configuration to output pandas
set_config(transform_output="pandas")

preprocessor=ColumnTransformer(
    transformers=[
        ('gender', gender_transformer, ['gender']),
        ('country', country_transformer, ['country']),
        ('credit_score', credit_score_transformer, ['credit_score']),
        # ('acc_balance', account_balance_transformer, ['acc_balance']),
        ('prod_count', prod_count_transformer, ['prod_count'])
    ],
    
    remainder='passthrough' 
)
# preprocessor.set_output(transform='pandas')

# Master Pipeline
final_pipeline=Pipeline([
    ('preprocessor',preprocessor),
    ('Scaler',StandardScaler())
])

X_train_processed = final_pipeline.fit_transform(X_train)
X_val_processed = final_pipeline.transform(X_val)

# Preprocessing Completes

## 🤖 Model Building Starts
Training and comparing multiple classification algorithms on the preprocessed data using class-imbalance-aware settings (`class_weight='balanced'`, `scale_pos_weight`) since the target is imbalanced.

In [ ]:
# Classification models and evaluation metrics used for baseline comparison.
# Importing all the necessary models and metrics
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier,GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.metrics import f1_score,roc_auc_score, accuracy_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

In [ ]:
# Training every baseline model on the same preprocessed data and scoring on F1, ROC-AUC, and accuracy.
# class_weight='balanced' / scale_pos_weight are used where supported to account for the target class imbalance.
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "KNN": KNeighborsClassifier(),
    "SVC": SVC(probability=True, class_weight='balanced'), 
    "Decision Tree": DecisionTreeClassifier(class_weight='balanced'),
    "Random Forest": RandomForestClassifier(class_weight='balanced'),
    "Extra Trees": ExtraTreesClassifier(class_weight='balanced'),
    "LightGBM": LGBMClassifier(scale_pos_weight=3, verbose=-1),
    "Gradient Boosting": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier(scale_pos_weight=3, eval_metric='logloss')
}

results=[]

for name,model in models.items():
    model.fit(X_train_processed,y_train)

    # Generating binary predictions and probability scores
    y_pred = model.predict(X_val_processed)
    y_prob = model.predict_proba(X_val_processed) [:,1]

    # Calculating Metrics

    f1 = f1_score(y_val,y_pred)
    roc_auc = roc_auc_score(y_val,y_prob) 
    acc= accuracy_score(y_val,y_pred)

    results.append(
        {
            'Model':model,
            'F1_Score':f1,
            'ROC_AUC_Score':roc_auc,
            'Accuracy_Score':acc
        }
    )
    

### 🏆 Primary Model Comparison
Ranking all baseline models by validation F1 score (a better metric than accuracy here, given the class imbalance).

In [ ]:
# Collating results into a dataframe and sorting by F1 score (most relevant metric given the class imbalance).
results_df = pd.DataFrame(results).sort_values(by="F1_Score", ascending=False).reset_index(drop=True)
display(results_df)

### 📤 Base Model Submission (Reference / Disabled)
Code for generating a baseline submission from the best untuned model. Kept commented out since this notebook proceeds straight to hyperparameter tuning before generating the final submission.

In [ ]:
# --- Reference only: baseline submission generation, left disabled since tuning happens next ---
# test_df=pd.read_csv('/kaggle/input/competitions/mlp-kaggle-assignment-2-may-2026-term/test.csv')

# test_df = proportional_imputation(test_df, 'country')

# submission_ids = test_df['id']
# X_test = test_df.drop(columns=['id', 'customer_id', 'last_name'])
# X_test_processed = final_pipeline.transform(X_test)

# best_baseline = models['LightGBM'] 
# test_predictions = best_baseline.predict(X_test_processed)

# submission = pd.DataFrame({
#     'id': submission_ids,
#     'exit_status': test_predictions
# })

# submission.to_csv('baseline_submission.csv', index=False)
# print("Baseline submission 'baseline_submission.csv' saved successfully!")
# display(submission.head())

## 🎯 Hyperparameter Tuning of the Top 3 Models
Tuning the strongest baseline performers — LightGBM, XGBoost, and (later) CatBoost/SVC — using `RandomizedSearchCV` with stratified cross-validation, scored on F1.

### 💡 LightGBM Tuning
Randomized search over tree depth, learning rate, leaf count, regularization, and `scale_pos_weight` to handle class imbalance.

In [ ]:
# Randomized hyperparameter search for LightGBM across depth, learning rate, leaves, regularization, and imbalance handling.


cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lgbm_grid = {
    'n_estimators': [100, 200, 400, 600],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7, -1],
    'num_leaves': [15, 31, 63, 127],
    'scale_pos_weight': [1, 2, 3, 4],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [0, 0.1, 1]
}
lgbm_tuned = RandomizedSearchCV(
    LGBMClassifier(verbose=-1, random_state=42),
    param_distributions=lgbm_grid,
    n_iter=40,
    scoring='f1',
    cv=cv_strategy,
    random_state=42,
    n_jobs=-1
)
lgbm_tuned.fit(X_train_processed, y_train)
print(f"Best LightGBM Parameters: {lgbm_tuned.best_params_}")
print(f"Best LightGBM Validation F1: {lgbm_tuned.best_score_:.4f}")


### 💡 XGBoost Tuning
Randomized search over depth, learning rate, subsampling, and regularization parameters.

In [ ]:
# --- First LightGBM/XGBoost search attempt, superseded by the cell below; kept for reference ---
# cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# xgb_grid = {
#     'n_estimators': [100, 200, 400, 600],
#     'learning_rate': [0.01, 0.05, 0.1, 0.2],
#     'max_depth': [3, 5, 7, 9],
#     'min_child_weight': [1, 3, 5],
#     'subsample': [0.8, 1.0],
#     'colsample_bytree': [0.8, 1.0],
#     'scale_pos_weight': [1, 2, 3, 4],
#     'reg_alpha': [0, 0.1, 1],
#     'reg_lambda': [1, 1.5, 2]
# }
# xgb_tuned = RandomizedSearchCV(
#     XGBClassifier(eval_metric='logloss', random_state=42),
#     param_distributions=xgb_grid,
#     n_iter=40,
#     scoring='f1',
#     cv=cv_strategy,
#     random_state=42,
#     n_jobs=-1
# )
# xgb_tuned.fit(X_train_processed, y_train)
# print(f"\nBest XGBoost Parameters: {xgb_tuned.best_params_}")
# print(f"Best XGBoost Validation F1: {xgb_tuned.best_score_:.4f}")

In [ ]:
# Randomized hyperparameter search for XGBoost (second/final attempt) across depth, learning rate, subsampling, and regularization.
#xgb 2 

xgb_grid = {
    'n_estimators': [100, 200, 400, 600],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7, 9],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'scale_pos_weight': [1, 2, 3, 4],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [1, 1.5, 2]
}
xgb_tuned = RandomizedSearchCV(
    XGBClassifier(eval_metric='logloss', random_state=42),
    param_distributions=xgb_grid,
    n_iter=40,
    scoring='f1',
    cv=cv_strategy,
    random_state=42,
    n_jobs=-1
)
xgb_tuned.fit(X_train_processed, y_train)
print(f"Best XGBoost Parameters: {xgb_tuned.best_params_}")
print(f"Best XGBoost Validation F1: {xgb_tuned.best_score_:.4f}")

### 💡 SVC Tuning
SVC training is expensive on the full dataset, so the search is run on a random subsample of up to 15,000 rows for speed, then the best parameters are refit on the full training set.

In [ ]:
# For speed, SVC hyperparameter search runs on a random subsample (max 15k rows) before refitting the best params on the full training set.

if X_train_processed.shape[0] > 15000:
    rng = np.random.RandomState(42)
    idx = rng.choice(X_train_processed.shape[0], 15000, replace=False)
    X_search = X_train_processed.iloc[idx]
    y_search = y_train.iloc[idx] if hasattr(y_train, 'iloc') else y_train[idx]
else:
    X_search, y_search = X_train_processed, y_train

svc_grid = {
    'C': [0.1, 1, 10, 50, 100],
    'gamma': [0.001, 0.005, 0.01, 0.05, 0.1, 'scale', 'auto'],
    'kernel': ['rbf']  
}
svc_search = RandomizedSearchCV(
    SVC(probability=False, class_weight='balanced', random_state=42, cache_size=1000),
    param_distributions=svc_grid,
    n_iter=10,
    scoring='f1',
    cv=cv_strategy,
    random_state=42,
    n_jobs=-1
)
svc_search.fit(X_search, y_search)
print(f"\nBest SVC Parameters: {svc_search.best_params_}")
print(f"Best SVC Validation F1: {svc_search.best_score_:.4f}")

best_svc = SVC(**svc_search.best_params_, class_weight='balanced',
               probability=True, random_state=42, cache_size=1000)
best_svc.fit(X_train_processed, y_train)

## 🐈 Training CatBoost
CatBoost handles categorical features natively, so a separate, lighter preprocessing pipeline is used here — imputation only, without one-hot encoding or scaling — and `gender`/`country` are passed to CatBoost as declared categorical features.

In [ ]:
# Lighter preprocessing pipeline for CatBoost: imputation only (no one-hot encoding or scaling), since CatBoost
# handles categorical features natively via the cat_features argument.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn import set_config

set_config(transform_output="pandas")

# Same imputation strategies as our main pipeline, but NO OneHotEncoder, NO Scaler
gender_transformer_cb = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent'))
])

country_transformer_cb = Pipeline([
    # no imputer here — country already imputed via proportional_imputation() before split
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('passthrough', 'passthrough')
])

credit_score_transformer_cb = Pipeline([
    ('impute', SimpleImputer(strategy='mean'))
])

# no acc_balance transformer needed — already imputed via impute_acc_balance() before split

prod_count_transformer_cb = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent'))
])

preprocessor_cb = ColumnTransformer(
    transformers=[
        ('gender', gender_transformer_cb, ['gender']),
        ('country', country_transformer_cb, ['country']),
        ('credit_score', credit_score_transformer_cb, ['credit_score']),
        ('prod_count', prod_count_transformer_cb, ['prod_count'])
    ],
    remainder='passthrough'
)

X_train_cb = preprocessor_cb.fit_transform(X_train)
X_val_cb = preprocessor_cb.transform(X_val)

X_train_cb.columns = [c.split('__')[-1] for c in X_train_cb.columns]
X_val_cb.columns = [c.split('__')[-1] for c in X_val_cb.columns]

print(X_train_cb.dtypes)
print(X_train_cb[['gender', 'country']].head())

In [ ]:
# Re-applying the CatBoost-specific preprocessing and running a randomized hyperparameter search with gender/country
# passed as native categorical features.
X_train_cb = preprocessor_cb.fit_transform(X_train)
X_val_cb = preprocessor_cb.transform(X_val)
X_train_cb.columns = [c.split('__')[-1] for c in X_train_cb.columns]
X_val_cb.columns = [c.split('__')[-1] for c in X_val_cb.columns]

from catboost import CatBoostClassifier

cat_features = ['gender', 'country']

cb_grid = {
    'iterations': [200, 400, 600],
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'l2_leaf_reg': [1, 3, 5, 7],
    'scale_pos_weight': [1, 2, 3, 4]
}
cb_tuned = RandomizedSearchCV(
    CatBoostClassifier(verbose=0, random_state=42),
    param_distributions=cb_grid,
    n_iter=25,
    scoring='f1',
    cv=cv_strategy,
    random_state=42,
    n_jobs=-1
)
cb_tuned.fit(X_train_cb, y_train, cat_features=cat_features)
print(f"Best CatBoost Parameters: {cb_tuned.best_params_}")
print(f"Best CatBoost Validation F1: {cb_tuned.best_score_:.4f}")

## 📤 Submission After Hyperparameter Tuning (Reference / Disabled)
Code for generating a submission from the best tuned model at this stage. Kept commented out — the final submission further down retrains the chosen model on the **full** training data first.

In [ ]:
# --- Reference only: submission generation right after tuning, superseded by the full-data retrain below ---
# test_df=pd.read_csv('/kaggle/input/competitions/mlp-kaggle-assignment-2-may-2026-term/test.csv')

# test_df = proportional_imputation(test_df, 'country')

# submission_ids = test_df['id']
# X_test = test_df.drop(columns=['id', 'customer_id', 'last_name'])
# X_test_processed = final_pipeline.transform(X_test)

# best_tuned = xgb_tuned
# test_predictions = best_tuned.predict(X_test_processed)

# submission = pd.DataFrame({
#     'id': submission_ids,
#     'exit_status': test_predictions
# })

# submission.to_csv('best_tuned.csv', index=False)
# print("Best tuned submission  'best_tuned.csv' saved successfully!")
# display(submission.head())

In [ ]:
# Comparing validation F1 scores of the three tuned models (LightGBM, XGBoost, CatBoost) side by side.
lgbm_val_f1 = f1_score(y_val, lgbm_tuned.best_estimator_.predict(X_val_processed))
xgb_val_f1 = f1_score(y_val, xgb_tuned.best_estimator_.predict(X_val_processed))
cb_val_f1 = f1_score(y_val, cb_tuned.best_estimator_.predict(X_val_cb))

comparison = pd.DataFrame({
    'Model': ['LightGBM', 'XGBoost', 'CatBoost'],
    'Val_F1': [lgbm_val_f1, xgb_val_f1, cb_val_f1]
}).sort_values('Val_F1', ascending=False).reset_index(drop=True)

print(comparison.to_string(index=False))

### 🏁 Final Submission — Trained on Full Training Data
Combining the train and validation splits back together, applying the same cleaning/imputation/feature-engineering steps to the test set, and generating the final predictions from the best-performing tuned model.

In [ ]:
# Recombining train + validation into the full training set, and applying the exact same cleaning/imputation/
# feature-engineering steps (using train-derived statistics) to the raw test set.
X_full = pd.concat([X_train, X_val], axis=0)
y_full = pd.concat([y_train, y_val], axis=0)

test_df = pd.read_csv('/kaggle/input/competitions/mlp-kaggle-assignment-2-may-2026-term/test.csv')
test_df = proportional_imputation(test_df, 'country')
test_df = impute_acc_balance(test_df, zero_rate, nonzero_values, seed=42)  # same train-derived stats
test_df = engineer_features(test_df, prod_count_mode)

submission_ids = test_df['id']
X_test = test_df.drop(columns=['id', 'customer_id', 'last_name'])

In [ ]:
# Fitting the final preprocessing pipeline on the full training data and transforming the test set with it.
X_full_processed = final_pipeline.fit_transform(X_full)
X_test_processed = final_pipeline.transform(X_test)

In [ ]:
# Refitting the best tuned model (LightGBM) on 100% of the training data and generating the final test predictions/submission file.


final_lgbm = lgbm_tuned.best_estimator_
final_lgbm.fit(X_full_processed, y_full)
lgbm_test_preds = final_lgbm.predict(X_test_processed)

submission = pd.DataFrame({'id': submission_ids, 'exit_status': lgbm_test_preds})
submission.to_csv('submission.csv', index=False)
print(submission['exit_status'].value_counts(normalize=True))
display(submission.head())

In [ ]:
# --- Reference only: equivalent submission steps using the tuned XGBoost model instead ---
# --- XGB submission ---
# final_xgb = xgb_tuned.best_estimator_
# final_xgb.fit(X_full_processed, y_full)
# xgb_test_preds = final_xgb.predict(X_test_processed)

# submission_xgb = pd.DataFrame({'id': submission_ids, 'exit_status': xgb_test_preds})
# submission_xgb.to_csv('submission.csv', index=False)
# print(submission_xgb['exit_status'].value_counts(normalize=True))

In [ ]:
# --- Reference only: equivalent submission steps using the tuned CatBoost model (requires the _cb preprocessed versions) ---
# --- CatBoost submission (needs the _cb versions of full/test data) ---
# X_full_cb = preprocessor_cb.fit_transform(X_full)
# X_full_cb.columns = [c.split('__')[-1] for c in X_full_cb.columns]

# X_test_cb = preprocessor_cb.transform(X_test)
# X_test_cb.columns = [c.split('__')[-1] for c in X_test_cb.columns]

# final_cb = cb_tuned.best_estimator_
# final_cb.fit(X_full_cb, y_full, cat_features=cat_features)
# cb_test_preds = final_cb.predict(X_test_cb)

# submission_cb = pd.DataFrame({'id': submission_ids, 'exit_status': cb_test_preds})
# submission_cb.to_csv('best_cb2.csv', index=False)
# print(submission_cb['exit_status'].value_counts(normalize=True))